In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# CrysText Training Notebook
# ===========================
# Fine-tunes Mistral-7B-v0.3 with QLoRA on MP-20 dataset
# Run on Kaggle with T4 GPU (16GB VRAM)
#
# BEFORE RUNNING:
# 1. Enable GPU in Kaggle → Settings → Accelerator → GPU T4
# 2. Enable internet in Kaggle → Settings → Internet → On
# 3. Add your HuggingFace token in Kaggle → Add-ons → Secrets → HF_TOKEN
#
# MODEL OUTPUT: rakshitha9/crystext-mistral-10k on HuggingFace

In [ ]:
!pip install unsloth -q
print("✅ Unsloth done!")

In [ ]:
!pip install trl peft accelerate -q
print("✅ TRL done!")

In [ ]:
!pip install bitsandbytes -q
print("✅ Bitsandbytes done!")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ HuggingFace logged in!")

In [ ]:
import os
import pandas as pd

os.system("git clone https://github.com/jiaor17/DiffCSP.git")

train_df = pd.read_csv('DiffCSP/data/mp_20/train.csv')
val_df   = pd.read_csv('DiffCSP/data/mp_20/val.csv')

# Session 3 — rows 0 to 10k (starting fresh)
train_df = train_df.iloc[0:10000].reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print("✅ Data loaded!")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "mistralai/Mistral-7B-v0.3",
    max_seq_length = 1308,
    dtype = None,
    load_in_4bit = True,
    token = hf_token,
)
print("✅ Base Mistral loaded!")

In [ ]:
# SESSION SYSTEM:
# Session 3 = rows 0:10000     (DONE — loss 0.2466)
# Session 4 = rows 10000:20000 (PENDING)
# Session 5 = rows 20000:27136 (PENDING)
# To run Session 4, change iloc[0:10000] to iloc[10000:20000]
# To run Session 5, change iloc[0:10000] to iloc[20000:27136]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
model.print_trainable_parameters()
print("✅ LoRA adapters attached!")

In [ ]:
# Cell 8 — Prepare dataset
from datasets import Dataset

EOS_TOKEN = tokenizer.eos_token

def formatting_func(examples):
    inputs  = examples["input"]
    outputs = examples["output"]
    texts   = []
    for inp, out in zip(inputs, outputs):
        text = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Generate CIF for the given material description

### Input:
{inp}

### Response:
{out}""" + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

train_data = Dataset.from_dict({
    "input":  [f"Material composition is {row['pretty_formula']}. It has a space group number {row['spacegroup.number']}." for _, row in train_df.iterrows()],
    "output": train_df["cif"].tolist()
})

val_data = Dataset.from_dict({
    "input":  [f"Material composition is {row['pretty_formula']}. It has a space group number {row['spacegroup.number']}." for _, row in val_df.iterrows()],
    "output": val_df["cif"].tolist()
})

train_data = train_data.map(formatting_func, batched=True)
val_data   = val_data.map(formatting_func, batched=True)

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print("✅ Dataset ready!")

In [ ]:
# Cell 9 — Setup Trainer
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_data,
    eval_dataset  = val_data,
    args = SFTConfig(
        dataset_text_field        = "text",
        max_seq_length            = 1308,
        dataset_num_proc          = 2,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps              = 50,
        num_train_epochs          = 1,
        learning_rate             = 2e-4,
        fp16                      = not is_bfloat16_supported(),
        bf16                      = is_bfloat16_supported(),
        logging_steps             = 50,
        optim                     = "adamw_8bit",
        weight_decay              = 0.01,
        lr_scheduler_type         = "cosine",
        seed                      = 42,
        output_dir                = "/kaggle/working/outputs",
    ),
)
print("✅ Trainer ready!")

In [ ]:
trainer_stats = trainer.train()
print("✅ Training complete!")
print(f"⏱️ Time: {trainer_stats.metrics['train_runtime']/60:.2f} minutes")
print(f"📉 Loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# Cell 10 removed — had hardcoded token, unsafe for Git
# See Cell 11 for the correct push method using Kaggle secrets

In [ ]:
# Cell 11 — Push to HuggingFace IMMEDIATELY
print("🚀 Pushing to HuggingFace now...")

model.push_to_hub(
    "rakshitha9/crystext-mistral-10k",
    token   = hf_token,
    private = True
)
tokenizer.push_to_hub(
    "rakshitha9/crystext-mistral-10k",
    token   = hf_token,
    private = True
)

print("✅ Model safely pushed to HuggingFace!")
print("🔗 https://huggingface.co/rakshitha9/crystext-mistral-10k")